# Day 2: Building a Real-World Telecom RAG System

## 1. The Business Problem: SLA Breaches & High AHT
Telecom customer service agents are required to follow complex Service Level Agreements (SLAs). When a customer complains about an internet outage, the agent must check a 500-page internal manual to determine if they are authorized to dispatch a field technician and offer compensation.

Searching this manual manually takes **5 minutes**, increasing Average Handling Time (AHT) and costing the company millions.

<img src="../RAG_SLA_Business_Impact.png" alt="Business Impact" width="900"/>

**Our Goal:** Build a Retrieval-Augmented Generation (RAG) system that instantly reads the manual and answers the agent's question based on exact internal policies.

## 2. Environment Setup & Data Ingestion
Install and import necessary libraries, load the internal knowledge base text file (`../data/Telecom_Internal_KB.txt`), and split it into chunks.

In [ ]:
# Install required libraries
%pip install -q -U \
    langchain \
    langchain-community \
    langchain-core \
    langchain-google-genai \
    langchain-huggingface \
    sentence-transformers \
    faiss-cpu \
    python-dotenv \
    tqdm

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

# Multilingual embeddings (Crucial for matching Arabic queries to English text)
print("Loading local embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Load the Knowledge Base
print("Loading Knowledge Base...")
loader = TextLoader('../data/Telecom_Internal_KB.txt', encoding='utf-8')
documents = loader.load()
print(f"✅ Successfully loaded {len(documents)} document(s).")

In [5]:
# Split the text into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(documents)

print(f"✅ Loaded {len(documents)} document.")
print(f"✅ Split into {len(chunks)} chunks.")
print(f"🔍 Sample chunk: \n{chunks[3].page_content}")

✅ Loaded 1 document.
✅ Split into 481 chunks.
🔍 Sample chunk: 
## 2. Hardware Specifications & Router Guides


## 3. Embeddings & Vector Database
Convert the chunks into vector embeddings and store them in a local Vector Store (e.g., ChromaDB or FAISS).

In [8]:
from langchain_community.vectorstores import FAISS
from tqdm import tqdm
import time

print(f"Starting ingestion of {len(chunks)} chunks into FAISS...")

# We ingest in batches to lower resource costs
batch_size = 50
vectorstore = None

for i in tqdm(range(0, len(chunks), batch_size), desc="Embedding & Indexing Chunks"):
    batch = chunks[i:i + batch_size]
    
    if vectorstore is None:
        # First batch initializes the FAISS index
        vectorstore = FAISS.from_documents(batch, embeddings)
    else:
        # Subsequent batches are added to the existing index
        vectorstore.add_documents(batch)
        

# Save the FAISS index locally so we don't have to pay/wait to re-embed later
vectorstore.save_local("faiss_telecom_index")
print("\n✅ Ingestion Complete. FAISS index saved locally.")

# Test if the multilingual retrieval actually works!
print("\nTesting semantic search (Arabic Query -> English Document)...")
test_query = "العميل بيشتكي إن لمبة الراوتر بتنور وتطفي بقالها ٣ أيام"
print(f"\n {test_query}")

results = vectorstore.similarity_search_with_score(test_query, k=10)
print("\n✅ Top match retrieved by FAISS:")
print("--------------------------------------------------")

# Unpack the tuple for the first result
best_doc, best_score = results[0]
print(f"Match Score (Lower distance is better): {best_score:.4f}")
print(best_doc.page_content)
print("--------------------------------------------------")
# Unpack the second result
second_doc, second_score = results[1]
print(f"Match Score: {second_score:.4f}")
print(second_doc.page_content)
print("--------------------------------------------------")



# Unpack the second result
second_doc, second_score = results[2]
print(f"Match Score: {second_score:.4f}")
print(second_doc.page_content)
print("--------------------------------------------------")



# Unpack the second result
second_doc, second_score = results[3]
print(f"Match Score: {second_score:.4f}")
print(second_doc.page_content)
print("--------------------------------------------------")

Starting ingestion of 481 chunks into FAISS...

✅ Ingestion Complete. FAISS index saved locally.

Testing semantic search (Arabic Query -> English Document)...

 العميل بيشتكي إن لمبة الراوتر بتنور وتطفي بقالها ٣ أيام

✅ Top match retrieved by FAISS:
--------------------------------------------------
Match Score (Lower distance is better): 12.1444
## 1. General Service Level Agreement (SLA) & Dispatch Policies
If a customer reports an internet outage (DSL blinking or no sync):
- The L1 agent must first ensure the customer has restarted the router and checked internal wiring.
- If the issue persists for more than 24 hours, the L1 agent must escalate to the Central Exchange team.
- A Field Technician must be dispatched if the line noise margin is below 6dB or if the DSL light is completely off/blinking for 3 consecutive days.
--------------------------------------------------
Match Score: 12.6735
## 4. Cross-Department Escalation Matrix
- **Billing Issues:** Transfer to 111.
- **Fiber Op

## 4. Prompt Engineering & The RAG Chain
Write the System Prompt enforcing the LLM to act as a telecom support assistant. Inject the retrieved context and the customer's Egyptian Arabic ticket.

In [9]:
import os
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

print("Building the Prompt and Gemini RAG Chain...")

# Load variables from the .env file
load_dotenv()

# Set Gemini API Key
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

# Initialize the Gemini LLM
gemini_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# The System Prompt (Instructions + Context Injection)
template = """
أنت موظف خدمة عملاء في مزود خدمة إنترنت (ISP). 
مهمتك هي الرد على شكوى العميل باللغة العامية المصرية بطريقة مهذبة واحترافية.
تحذير هام: إياك أن تذكر أي اسم شركة اتصالات حقيقي (مثل اتصالات، فودافون، وي، إلخ) في ردك. قدم نفسك فقط كموظف خدمة عملاء فقط.
يجب عليك استخدام المعلومات الموجودة في (السياق الداخلي) فقط لحل المشكلة.
إذا كانت المشكلة تستدعي إرسال فني حسب القواعد، أخبر العميل بذلك بناءً على السياق.
السياق الداخلي (قوانين الشركة وخطوات الحل):
{context}
شكوى العميل:
{question}
الرد:
"""

prompt = PromptTemplate.from_template(template)
print(f"{prompt.template}")

# Convert FAISS vectorstore into a retriever (pulling top 20 chunks)
retriever = vectorstore.as_retriever(search_kwargs={"k": 20})

# Helper function to combine the retrieved chunks into one text block
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build the RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | gemini_llm | StrOutputParser()
)
print("✅ Gemini RAG Chain is ready!")


Building the Prompt and Gemini RAG Chain...

أنت موظف خدمة عملاء في مزود خدمة إنترنت (ISP). 
مهمتك هي الرد على شكوى العميل باللغة العامية المصرية بطريقة مهذبة واحترافية.
تحذير هام: إياك أن تذكر أي اسم شركة اتصالات حقيقي (مثل اتصالات، فودافون، وي، إلخ) في ردك. قدم نفسك فقط كموظف خدمة عملاء فقط.
يجب عليك استخدام المعلومات الموجودة في (السياق الداخلي) فقط لحل المشكلة.
إذا كانت المشكلة تستدعي إرسال فني حسب القواعد، أخبر العميل بذلك بناءً على السياق.
السياق الداخلي (قوانين الشركة وخطوات الحل):
{context}
شكوى العميل:
{question}
الرد:

✅ Gemini RAG Chain is ready!


## 5. Live Test: Solving the Egyptian Support Ticket
Test the pipeline with a realistic customer complaint.

In [10]:
print("Processing the ticket through Gemini...\n")

customer_ticket = """
أنا دافع الفاتورة من يومين أونلاين والفلوس اتخصمت من الفيزا، 
لكن النت لسه مرجعش لحد دلوقتي ومكتوبلي إن الخدمة موقوفة!
"""

print("Agent AI Response (Gemini):")
print("--------------------------------------------------")

# This sends the ticket to the retriever, formats the prompt, and gets the answer from Gemini
response = rag_chain.invoke(customer_ticket)

print(response)
print("--------------------------------------------------")

Processing the ticket through Gemini...

Agent AI Response (Gemini):
--------------------------------------------------
أهلاً بحضرتك، أنا من خدمة عملاء مزود خدمة الإنترنت.

أنا متفهم جداً للموقف اللي حضرتك فيه وإن النت مش شغال بالرغم من دفع الفاتورة والفلوس اتخصمت من الفيزا.

بالنسبة لمشكلة دفع الفاتورة وتوقف الخدمة، دي بتحتاج تحويل لقسم الفواتير عشان يتأكدوا من وصول المبلغ وتفعيل الخدمة لحضرتك في أسرع وقت.

بعد إذنك، هحول حضرتك على طول لقسم الفواتير عشان يراجعوا الموضوع ويحلوا المشكلة دي. بعتذر جداً عن أي إزعاج حصل لحضرتك.
--------------------------------------------------


In [11]:
print("Processing the ticket through Gemini...\n")

customer_ticket = """
النت شغال بس بطيء جداً وبيظهرلي رسالة على الشاشة فيها كود الخطأ E-204.
أعمل إيه عشان أحل المشكلة دي؟
"""

print("Agent AI Response (Gemini):")
print("--------------------------------------------------")

# This sends the ticket to the retriever, formats the prompt, and gets the answer from Gemini
response = rag_chain.invoke(customer_ticket)

print(response)
print("--------------------------------------------------")

Processing the ticket through Gemini...

Agent AI Response (Gemini):
--------------------------------------------------
أهلاً بحضرتك، أنا موظف خدمة العملاء من مزود خدمة الإنترنت.

متفهم جداً إن حضرتك بتواجه مشكلة بطء في الإنترنت وظهور كود الخطأ E-204. الكود ده بيشير عادةً لمشكلة في جودة الخط.

ممكن بعد إذنك تجرب خطوة بسيطة وإن شاء الله تحل المشكلة:
حضرتك هتحتاج تغير إعدادات الـ DNS في جهازك لـ **8.8.8.8**.

ياريت تجرب الخطوة دي وتبلغنا بالنتيجة. لو محتاج أي مساعدة في طريقة تغيير الـ DNS، ممكن أوضح لحضرتك الخطوات بالتفصيل. إحنا موجودين للمساعدة في أي وقت.
--------------------------------------------------
